# DuckDB Check Queries for Curated Parquet

This notebook verifies that the curated Parquet datasets are:
- readable via DuckDB directly (no pre-loading into pandas)
- complete enough for analytics
- internally consistent (duplicates, basic validity constraints)
- suitable for basic downstream analytics (returns, rolling volatility)

Outputs:
- `reports/duckdb_check_summary.csv`

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone

import duckdb
import pandas as pd

In [2]:
#----Parameters (edit if needed) ----
CURATED_DAILY_GLOB = "../data/curated/bars_daily/**/*.parquet"
CURATED_1M_GLOB = "../data/curated/bars_1m/**/*.parquet"

#Output report directory
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_OUT = REPORTS_DIR / "duckdb_check_summary.csv"

#Optional: persist a DuckDB database file instead of in-memory
USE_PERSISTENT_DB = False
DUCKDB_DB_PATH = "duckdb.db" # used only if USE_PERSISTENT=True

Connect to DuckDB

In [3]:
con = duckdb.connect(DUCKDB_DB_PATH) if USE_PERSISTENT_DB else duckdb.connect()

#Helpful options
con.execute("SET TimeZone='UTC';")
con.execute("PRAGMA enable_progress_bar=false;")

print("DuckDB version:", con.execute("SELECT version()").fetchone()[0])

DuckDB version: v1.4.4


Dataset Discovery: do files exist?

In [4]:
from glob import glob

daily_files = glob(CURATED_DAILY_GLOB, recursive=True)
min_files = glob(CURATED_1M_GLOB, recursive=True)

print(f"Daily files found: {len(daily_files)}")
print(f"1-Min files found: {len(min_files)}")

if len(daily_files) == 0:
    print("WARNING: No daily parquet files found.  Check CURATED_DAILY_GLOB.")
if len(min_files) == 0:
    print("WARNING:  No 1-minute parquet files found (this may be unexpected).")

Daily files found: 150
1-Min files found: 3150


Schema Check (Daily)

In [5]:
daily_schema = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{CURATED_DAILY_GLOB}');
""").df()

daily_schema

,column_name,column_type,null,key,default,extra
0,ts_utc,TIMESTAMP WITH TIME ZONE,YES,None,None,None
1,open,DOUBLE,YES,None,None,None
2,high,DOUBLE,YES,None,None,None
3,low,DOUBLE,YES,None,None,None
4,close,DOUBLE,YES,None,None,None
5,volume,BIGINT,YES,None,None,None
6,source,VARCHAR,YES,None,None,None
7,timeframe,VARCHAR,YES,None,None,None
8,date,VARCHAR,YES,None,None,None
9,symbol,VARCHAR,YES,None,None,None


Schema Check (1-Min if present)

In [6]:
if len(min_files) > 0:
    min_schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{CURATED_1M_GLOB}');
    """).df()
    print(min_schema)
else:
    print("No 1-minute dataset present; skipping schema check.")

   column_name               column_type null   key default extra
0       ts_utc  TIMESTAMP WITH TIME ZONE  YES  None    None  None
1         open                    DOUBLE  YES  None    None  None
2         high                    DOUBLE  YES  None    None  None
3          low                    DOUBLE  YES  None    None  None
4        close                    DOUBLE  YES  None    None  None
5       volume                    BIGINT  YES  None    None  None
6       source                   VARCHAR  YES  None    None  None
7    timeframe                   VARCHAR  YES  None    None  None
8         year                    BIGINT  YES  None    None  None
9         date                      DATE  YES  None    None  None
10      symbol                   VARCHAR  YES  None    None  None


Row Counts + Coverage Table (Daily)

In [7]:
daily_coverage = con.execute(f"""
WITH base AS (
    SELECT
    symbol,
    CAST(ts_utc AS TIMESTAMP) AS ts_utc,
    CAST(timeframe as VARCHAR) AS timeframe
    FROM read_parquet('{CURATED_DAILY_GLOB}')
)
SELECT 
    symbol,
    MIN(ts_utc) AS min_ts,
    MAX(ts_utc) AS max_ts,
    COUNT(*)    AS n_rows,
    CAST(MAX(ts_utc) AS DATE) AS most_recent_date
FROM base
GROUP BY symbol
ORDER BY symbol;
""").df()

daily_coverage

,symbol,min_ts,max_ts,n_rows,most_recent_date
0,AAPL,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
1,ABBV,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
2,ADBE,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
3,AMD,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
4,AMZN,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
5,AVGO,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
6,BA,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
7,BAC,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
8,C,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31
9,CAT,2023-12-01 05:00:00,2025-12-31 05:00:00,63,2025-12-31


Total Row Count

In [8]:
daily_total_rows = con.execute(f"""
SELECT COUNT(*) as total_rows
FROM read_parquet('{CURATED_DAILY_GLOB}');
""").fetchone()[0]

daily_total_rows

3150

Duplicate Detection (Daily)

In [9]:
daily_dupes = con.execute(f"""
WITH base AS (
SELECT
  symbol,
  CAST(ts_utc AS TIMESTAMP) AS ts_utc,
  CAST(timeframe AS VARCHAR) AS timeframe
  FROM read_parquet('{CURATED_DAILY_GLOB}')
)
SELECT 
  symbol,
  ts_utc,
  timeframe,
  COUNT(*) AS dup_count
FROM base
GROUP BY symbol, ts_utc, timeframe
HAVING COUNT(*) > 1
ORDER BY dup_count DESC, symbol, ts_utc
LIMIT 200;
""").df()

daily_dupes

,symbol,ts_utc,timeframe,dup_count


Basic Validity Checks (Daily)
Counts rows where:
- close <= 0
- volume < 0
- high < GREATEST(open,close)
- low > LEAST(open,close)

In [10]:
daily_validity = con.execute(f"""
SELECT
  SUM(CASE WHEN close <= 0 THEN 1 ELSE 0 END) AS close_le_zero,
  SUM(CASE WHEN volume < 0 THEN 1 ELSE 0 END) AS volume_lt_zero,
  SUM(CASE WHEN high < GREATEST(open,close) THEN 1 ELSE 0 END) AS high_lt_greatest_open_close,
  SUM(CASE WHEN low > LEAST(open,close) THEN 1 ELSE 0 END) AS low_gt_least_open_close,
  COUNT(*) AS total_rows_checked
FROM read_parquet('{CURATED_DAILY_GLOB}');
""").df()

daily_validity

,close_le_zero,volume_lt_zero,high_lt_greatest_open_close,low_gt_least_open_close,total_rows_checked
0,0.0,0.0,0.0,0.0,3150


Simple Analytics: Daily Returns per Symbol (LAG)

In [12]:
daily_returns_sample = con.execute(f"""
WITH base AS (
SELECT
  symbol,
  CAST(ts_utc AS TIMESTAMP) AS ts_utc,
  close
FROM read_parquet('{CURATED_DAILY_GLOB}')
),
rets AS (
  SELECT 
    symbol,
    ts_utc,
    close,
    LAG(close) OVER (PARTITION BY symbol ORDER BY ts_utc) AS prev_close
  FROM base
)
SELECT
  symbol,
  ts_utc,
  close,
  prev_close,
  CASE
    WHEN prev_close IS NULL OR prev_close = 0 THEN NULL
    ELSE (close / prev_close) - 1
  END AS daily_return 
FROM rets
WHERE prev_close IS NOT NULL
ORDER BY ts_utc DESC, symbol
LIMIT 50;
""").df()

daily_returns_sample

,symbol,ts_utc,close,prev_close,daily_return
0,AAPL,2025-12-31 05:00:00,272.035,273.050,-0.003717
1,ABBV,2025-12-31 05:00:00,228.480,229.695,-0.005290
2,ADBE,2025-12-31 05:00:00,350.030,352.490,-0.006979
3,AMD,2025-12-31 05:00:00,214.120,215.400,-0.005942
4,AMZN,2025-12-31 05:00:00,230.850,232.490,-0.007054
5,AVGO,2025-12-31 05:00:00,346.220,349.910,-0.010546
6,BA,2025-12-31 05:00:00,217.100,218.490,-0.006362
7,BAC,2025-12-31 05:00:00,55.000,55.280,-0.005065
8,C,2025-12-31 05:00:00,116.680,117.205,-0.004479
9,CAT,2025-12-31 05:00:00,572.835,577.380,-0.007872


Rolling Volatility + "Top 10 Most Volatile Last 60 Trading Days"

Uses `STDDEV_SAMP` over a rolling window (20-day) and looks back ~ 60 trading days

In [13]:
top10_volatile = con.execute(f"""
WITH base AS (
  SELECT
    symbol,
    CAST(ts_utc AS TIMESTAMP) AS ts_utc,
    close
  FROM read_parquet('{CURATED_DAILY_GLOB}')
),
rets AS (
  SELECT
    symbol,
    ts_utc,
    close,
    LAG(close) OVER (PARTITION BY symbol ORDER BY ts_utc) AS prev_close
  FROM base
),
r AS (
  SELECT
    symbol,
    ts_utc,
    CASE
      WHEN prev_close IS NULL OR prev_close = 0 THEN NULL
      ELSE (close / prev_close) - 1
    END AS ret
  FROM rets
),
roll AS (
  SELECT
    symbol,
    ts_utc,
    ret,
    STDDEV_SAMP(ret) OVER (
      PARTITION BY symbol
      ORDER BY ts_utc
      ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
    ) AS vol_20d
  FROM r
),
last60 AS (
  SELECT *
  FROM roll
  WHERE ts_utc >= (
    SELECT MAX(CAST(ts_utc AS TIMESTAMP)) - INTERVAL 90 DAY
    FROM read_parquet('{CURATED_DAILY_GLOB}')
  )
)
SELECT
  symbol,
  AVG(vol_20d) AS avg_20d_vol_last_window,
  MAX(vol_20d) AS max_20d_vol_last_window,
  COUNT(*)     AS n_obs
FROM last60
WHERE vol_20d IS NOT NULL
GROUP BY symbol
ORDER BY avg_20d_vol_last_window DESC
LIMIT 10;
""").df()

top10_volatile

,symbol,avg_20d_vol_last_window,max_20d_vol_last_window,n_obs
0,INTC,0.207243,0.225996,22
1,NFLX,0.179338,0.196739,22
2,AMD,0.170530,0.186596,22
3,GE,0.150407,0.165231,22
4,AVGO,0.144622,0.159702,22
5,GOOGL,0.137165,0.149884,22
6,CAT,0.117828,0.128135,22
7,C,0.094308,0.103481,22
8,RTX,0.092736,0.101678,22
9,JNJ,0.087297,0.095154,22


Repeat Key Checks for 1-Min

In [14]:
if len(min_files) > 0:
    min_total_rows = con.execute(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{CURATED_1M_GLOB}');
    """).df()

    min_coverage = con.execute(f"""
    WITH base AS (
      SELECT
        symbol,
        CAST(ts_utc AS TIMESTAMP) AS ts_utc,
        CAST(timeframe AS VARCHAR) AS timeframe
      FROM read_parquet('{CURATED_1M_GLOB}')
    )
    SELECT
      symbol,
      MIN(ts_utc) AS min_ts,
      MAX(ts_utc) AS max_ts,
      COUNT(*)    AS n_rows,
      CAST(MAX(ts_utc) AS DATE) AS most_recent_date
    FROM base
    GROUP BY symbol
    ORDER BY symbol;
    """).df()

    min_dupes = con.execute(f"""
    WITH base AS (
      SELECT
        symbol,
        CAST(ts_utc AS TIMESTAMP) AS ts_utc,
        CAST(timeframe AS VARCHAR) AS timeframe
      FROM read_parquet('{CURATED_1M_GLOB}')
    )
    SELECT
      symbol, ts_utc, timeframe, COUNT(*) AS dup_count
    FROM base
    GROUP BY symbol, ts_utc, timeframe
    HAVING COUNT(*) > 1
    ORDER BY dup_count DESC
    LIMIT 200;
    """).df()

    min_validity = con.execute(f"""
    SELECT
      SUM(CASE WHEN close <= 0 THEN 1 ELSE 0 END) AS close_le_zero,
      SUM(CASE WHEN volume < 0 THEN 1 ELSE 0 END) AS volume_lt_zero,
      SUM(CASE WHEN high < GREATEST(open, close) THEN 1 ELSE 0 END) AS high_lt_greatest_open_close,
      SUM(CASE WHEN low  > LEAST(open, close)     THEN 1 ELSE 0 END) AS low_gt_least_open_close,
      COUNT(*) AS total_rows_checked
    FROM read_parquet('{CURATED_1M_GLOB}');
    """).df()

    display(min_total_rows)
    display(min_coverage.head(25))
    display(min_dupes)
    display(min_validity)
else:
    print("No 1-minute dataset present; skipping 1-minute checks.")

,total_rows
0,1073081


,symbol,min_ts,max_ts,n_rows,most_recent_date
0,AAPL,2025-11-03 13:51:00,2026-02-03 21:41:00,24267,2026-02-03
1,ABBV,2025-11-03 14:30:00,2026-02-03 20:59:00,17518,2026-02-03
2,ADBE,2025-11-03 14:30:00,2026-02-03 20:59:00,19800,2026-02-03
3,AMD,2025-11-03 13:01:00,2026-02-03 21:59:00,23817,2026-02-03
4,AMZN,2025-11-03 14:00:00,2026-02-03 20:59:00,24184,2026-02-03
5,AVGO,2025-11-03 13:37:00,2026-02-03 21:59:00,24456,2026-02-03
6,BA,2025-11-03 14:30:00,2026-02-03 20:59:00,19737,2026-02-03
7,BAC,2025-11-03 14:30:00,2026-02-03 20:59:00,24059,2026-02-03
8,C,2025-11-03 14:31:00,2026-02-03 20:59:00,23161,2026-02-03
9,CAT,2025-11-03 14:30:00,2026-02-03 20:59:00,17455,2026-02-03


,symbol,ts_utc,timeframe,dup_count


,close_le_zero,volume_lt_zero,high_lt_greatest_open_close,low_gt_least_open_close,total_rows_checked
0,0.0,0.0,0.0,0.0,1073081


Exportable Output: reports/duckdb_sanity_summary.csv

In [15]:
run_ts = datetime.now(timezone.utc).isoformat()

summary_rows = []

# Daily summary
summary_rows.append({
    "run_ts_utc": run_ts,
    "dataset": "bars_daily",
    "parquet_glob": CURATED_DAILY_GLOB,
    "total_rows": daily_total_rows,
    "symbols": int(daily_coverage.shape[0]),
    "duplicates_found": int(daily_dupes.shape[0]),
    "close_le_zero": int(daily_validity["close_le_zero"].iloc[0]),
    "volume_lt_zero": int(daily_validity["volume_lt_zero"].iloc[0]),
    "high_lt_greatest_open_close": int(daily_validity["high_lt_greatest_open_close"].iloc[0]),
    "low_gt_least_open_close": int(daily_validity["low_gt_least_open_close"].iloc[0]),
})

# Optional 1m summary
if len(min_files) > 0:
    min_total_rows_val = con.execute(f"SELECT COUNT(*) FROM read_parquet('{CURATED_1M_GLOB}')").fetchone()[0]
    summary_rows.append({
        "run_ts_utc": run_ts,
        "dataset": "bars_1m",
        "parquet_glob": CURATED_1M_GLOB,
        "total_rows": int(min_total_rows_val),
        "symbols": int(min_coverage.shape[0]),
        "duplicates_found": int(min_dupes.shape[0]),
        "close_le_zero": int(min_validity["close_le_zero"].iloc[0]),
        "volume_lt_zero": int(min_validity["volume_lt_zero"].iloc[0]),
        "high_lt_greatest_open_close": int(min_validity["high_lt_greatest_open_close"].iloc[0]),
        "low_gt_least_open_close": int(min_validity["low_gt_least_open_close"].iloc[0]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_OUT, index=False)

summary_df, f"Wrote: {SUMMARY_OUT}"

(                         run_ts_utc     dataset  \
 0  2026-02-24T20:08:53.162019+00:00  bars_daily   
 1  2026-02-24T20:08:53.162019+00:00     bars_1m   
 
                               parquet_glob  total_rows  symbols  \
 0  ../data/curated/bars_daily/**/*.parquet        3150       50   
 1     ../data/curated/bars_1m/**/*.parquet     1073081       50   
 
    duplicates_found  close_le_zero  volume_lt_zero  \
 0                 0              0               0   
 1                 0              0               0   
 
    high_lt_greatest_open_close  low_gt_least_open_close  
 0                            0                        0  
 1                            0                        0  ,
 'Wrote: reports\\duckdb_check_summary.csv')

## Troubleshooting

### 1) Missing Parquet files
- Confirm ingestion wrote to:
  - `data/curated/bars_daily/`
  - `data/curated/bars_1m/` (optional)
- Confirm the glob patterns match your folder structure.
- On Windows, globbing is still fine in DuckDB as long as paths use `/` or are properly escaped.

### 2) Schema mismatch / missing columns
- Run the `DESCRIBE SELECT * FROM read_parquet(...)` cells.
- Ensure your canonical schema includes:
  - `symbol`, `ts_utc`, `open`, `high`, `low`, `close`, `volume`, `timeframe`
  - (optional) `source`, `date` partition helper

### 3) Timezone parsing issues
- This notebook enforces `SET TimeZone='UTC'`.
- If `ts_utc` is stored as string, cast it:
  - `CAST(ts_utc AS TIMESTAMP)`
- If you see weird offsets, inspect raw types via the schema output.

### 4) DuckDB can't read a parquet file
- One file may be corrupted or have a different schema.
- Try narrowing:
  - `read_parquet('data/curated/bars_daily/symbol=XYZ/**/*.parquet')`
- If necessary, isolate the offending partition and re-run ingestion for it.